## Building LeNet

- LeNet original model.

In [4]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms

torch.use_deterministic_algorithms(True)

In [7]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.cn1 = nn.Conv2d(3, 6, 5)
        self.cn2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = F.relu(self.cn1(x))
        x = F.max_pool2d(x, (2,2))
        x = F.relu(self.cn2(x))
        x = F.max_pool2d(x, (2,2))
        x = x.view(-1, self.flattened_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
    def flattened_features(self, x):
        size = x.size()[1:]
        num_feats = 1
        for s in size:
            num_feats *= s
        return num_feats
        
lenet = LeNet()
print(lenet)

LeNet(
  (cn1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (cn2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [ ]:
def train(net, train_loader, optim, epoch):
    # initialize loss
    loss_total = 0.0
    
    for i, data in enumerate(train_loader, 0):
        input, ground_truth = data
        
        # zero the parameter gradients
        optim.zero_grad()
        
        # forward pass, backward pass, optimization step
        output = net(input)
        loss = nn.CrossEntropyLoss()(output, ground_truth)
        loss.backward()
        optim.step()
        
        # update the loss
        loss_total += loss.item()
        
        # print loss statistics
        if(i + 1) % 1000 == 0:
            print('[Epoch number : %d, Mini-batches: %5d] loss: %.3f' %
                  (epoch + 1,
                   i + 1, 
                   loss_total / 200
                   )
                )
            loss_total = 0.0

In [10]:
def test(net, test_loader):
    success = 0
    counter = 0
    
    with torch.no_grad():
        for data in test_loader:
            im, ground_truth = data
            op = net(im)
            _, pred = torch.max(op, ground_truth)
            counter += ground_truth.size(0)
            success += (pred == ground_truth).sum().item()
            
            
    print('LeNet accuracy on 10000 images from test dataset: %d %%' %(
        100 * success/counter
    ))